In [0]:
# Databricks notebook source

# ==========================================================
# Silver Data Quality Checks
#
# Notebook: 02_silver_dq_checks
#
# Purpose:
# Validate Silver tables before Gold processing.
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, UTC

CATALOG = "crypto_pipeline"

DIM_TABLE = f"{CATALOG}.silver.dim_coin"
FACT_TABLE = f"{CATALOG}.silver.fact_coin_price"
DQ_TABLE = f"{CATALOG}.meta.dq_results"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create DQ Results Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {DQ_TABLE}

(

layer STRING,

check_name STRING,

status STRING,

failed_rows INT,

check_timestamp TIMESTAMP

)

USING DELTA

""")

# ==========================================================
# Read Tables
# ==========================================================

dim_df = spark.table(DIM_TABLE)
fact_df = spark.table(FACT_TABLE)

dq_logs = []

# ==========================================================
# Helper Function
# ==========================================================

def log_result(layer, check_name, failed_rows):

    status = "PASS" if failed_rows == 0 else "FAIL"

    dq_logs.append(

        Row(

            layer=layer,

            check_name=check_name,

            status=status,

            failed_rows=int(failed_rows),

            check_timestamp=datetime.now(UTC)

        )

    )

# ==========================================================
# CHECK 1
# Duplicate Observation
# ==========================================================

duplicate_observations = (

    fact_df

    .groupBy(

        "coin_id",

        "observation_ts"

    )

    .count()

    .filter("count > 1")

)

dup_count = duplicate_observations.count()

log_result(

    "silver",

    "Duplicate Observation",

    dup_count

)

# ==========================================================
# CHECK 2
# One Current Record
# ==========================================================

multiple_current = (

    dim_df

    .filter("is_current = true")

    .groupBy("coin_id")

    .count()

    .filter("count > 1")

)

current_count = multiple_current.count()

log_result(

    "silver",

    "Multiple Current Rows",

    current_count

)

# ==========================================================
# CHECK 3
# Null Prices
# ==========================================================

null_prices = (

    fact_df

    .filter(

        F.col("current_price").isNull()

    )

)

null_count = null_prices.count()

log_result(

    "silver",

    "Null Current Price",

    null_count

)

# ==========================================================
# CHECK 4
# Negative Prices
# ==========================================================

negative_prices = (

    fact_df

    .filter(

        F.col("current_price") < 0

    )

)

negative_count = negative_prices.count()

log_result(

    "silver",

    "Negative Price",

    negative_count

)

# ==========================================================
# CHECK 5
# Null Coin SK
# ==========================================================

null_sk = (

    fact_df

    .filter(

        F.col("coin_sk").isNull()

    )

)

null_sk_count = null_sk.count()

log_result(

    "silver",

    "Null Coin SK",

    null_sk_count

)

# ==========================================================
# CHECK 6
# Duplicate Surrogate Keys
# ==========================================================

duplicate_sk = (

    dim_df

    .groupBy("coin_sk")

    .count()

    .filter("count > 1")

)

dup_sk_count = duplicate_sk.count()

log_result(

    "silver",

    "Duplicate Coin SK",

    dup_sk_count

)

# ==========================================================
# Write Results
# ==========================================================

spark.createDataFrame(dq_logs).write.mode("append").saveAsTable(DQ_TABLE)

display(

    spark.table(DQ_TABLE)

    .orderBy(

        F.desc("check_timestamp")

    )

)

# ==========================================================
# Fail Pipeline if Needed
# ==========================================================

failed = sum(

    x.failed_rows

    for x in dq_logs

)

if failed > 0:

    raise Exception(

        f"Silver Data Quality Failed. Failed Checks = {failed}"

    )

print("Silver Data Quality Passed Successfully.")

layer,check_name,status,failed_rows,check_timestamp
silver,Duplicate Coin SK,PASS,0,2026-07-22T10:44:29.333Z
silver,Null Coin SK,PASS,0,2026-07-22T10:44:28.528Z
silver,Negative Price,PASS,0,2026-07-22T10:44:27.865Z
silver,Null Current Price,PASS,0,2026-07-22T10:44:27.381Z
silver,Multiple Current Rows,PASS,0,2026-07-22T10:44:26.898Z
silver,Duplicate Observation,PASS,0,2026-07-22T10:44:26.219Z


Silver Data Quality Passed Successfully.
